In [28]:
import json
import os
from tqdm import tqdm
import argparse
import pickle   

from FlagEmbedding import FlagReranker
from llama_index.llms.ollama import Ollama
from llama_index.core import Settings,VectorStoreIndex,PromptTemplate,StorageContext,load_index_from_storage
from llama_index.core.schema import TextNode
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.response_synthesizers import ResponseMode
from llama_index.embeddings.ollama import OllamaEmbedding
from util.kg_post_processor import NaivePostprocessor,KGRetrievePostProcessor,ngram_overlap,GraphFilterPostProcessor
from util.kg_response_synthesizer import get_response_synthesizer

from itext2kg.models import KnowledgeGraph

In [70]:
sub_kg_list = os.listdir("/home/jovyan/my_code/itext2kg/output_kg/AD1")

In [71]:
all_kg = {}
for pmid in sub_kg_list:
    with open(f"/home/jovyan/my_code/itext2kg/output_kg/AD1/{pmid}", 'rb') as f:
        kg = pickle.load(f)
        pmid = pmid.split(".")[0]
        all_kg[f"pmid{pmid}"] = kg

In [ ]:
doc2kg = {}

for k, v in all_kg.items():
    doc2kg[k] = {}
    for relation in v.relationships:
        sub_re = []
        sub_re.append([relation.startEntity.name, relation.name, relation.endEntity.name])
    
    doc2kg[k] = {"1": sub_re}
    
    for entity in v.entities:
        doc2kg[entity.name] = {}
        en_relation = []
        for relation in v.relationships:
            if entity.name == relation.startEntity.name or entity.name == relation.endEntity.name:
                en_relation.append([relation.startEntity.name, relation.name, relation.endEntity.name])
        doc2kg[entity.name] = {"1": en_relation}

In [73]:
doc2kg

{'pmid39652311': {'1': [['pmid39652311', 'reported', "alzheimer's disease"]]},
 "alzheimer's disease": {'1': [['abeta', 'linked_to', "alzheimer's disease"],
   ["alzheimer's disease", 'instance_of', 'neurodegenerative disorder'],
   ['pmid10516307', 'reported', "alzheimer's disease"]]},
 'polyphenols': {'1': [['polyphenols',
    'has_effect_against',
    "alzheimer's disease"]]},
 'pmid39855320': {'1': [['pmid39855320', 'reported', 'neuroinflammation'],
   ['pmid39855320', 'reported', "alzheimer's disease"],
   ['pmid39855320', 'reported', 'cognitive decline']]},
 'neuroinflammation': {'1': [['extra virgin olive oil',
    'treats',
    'neuroinflammation'],
   ['neuroinflammation', 'associated_with', "parkinson's disease"],
   ['hydroxytyrosol', 'treats', 'neuroinflammation'],
   ['pmid39002917', 'reported', 'neuroinflammation']]},
 'tau': {'1': [['tau', 'reported', 'neurodegenerative disorders'],
   ['tau', 'associated_with', 'organellar dysfunction'],
   ['tau', 'reported', 'tdp 43']

In [74]:
Settings.llm = Ollama(model="deepseek-r1:32b", base_url="http://127.0.0.1:11434")
Settings.embed_model = OllamaEmbedding(model_name="nomic-embed-text:latest", base_url="http://127.0.0.1:11434")

In [75]:
top_k=5
persist_dir=None
reranker='/home/jovyan/my_code/KG2RAG/bge-reranker-large'

In [76]:
"""
适用于单个问题的测试
:param question: 需要测试的问题
:param context: 相关背景信息，格式类似于 `data.json` 里的 'context'
:param doc2kg: 知识图谱数据，实体对应的三元组信息
:param top_k: 检索的top_k
:param persist_dir: 存储向量索引的目录
:param reranker: 重排序模型
"""

# 构造 `doc_chunks` 和 `chunks_index`
doc_chunks = []  
chunks_index = {}
ents = set()


for k, v in all_kg.items():
    for entity in v.entities:
        ents.add(entity.name)
        if entity.label == 'abstract':
            ent_name = entity.name
            ent_conext = entity.properties_info['context']
            
            if ent_name not in chunks_index:
                chunks_index[ent_name] = {}

            doc_chunk = TextNode(
                text=f'{ent_name}: {ent_conext}', 
                id_=f'{ent_name}',
                embedding=list(entity.properties.embeddings),
                )
            doc_chunks.append(doc_chunk)
            chunks_index[ent_name] = ent_conext

In [78]:
len(chunks_index)

102

In [38]:
doc_chunks

[TextNode(id_='pmid32760437', embedding=[0.0256652776, 0.025274751999999998, -0.141138424, -0.0830131368, 0.0262163206, -0.0360128694, -0.0045279526, 0.0112806292, -0.006953094200000001, -0.015124770799999998, -0.0025085911999999985, 0.0190403142, 0.094221424, 0.00026167399999999993, 0.045388665, -0.0367291462, -0.018544838600000002, -0.036569062400000005, -0.007371336199999998, 0.0240227838, -0.015834828199999998, -0.01979398764, -0.024728071199999997, -0.0234725482, 0.134340412, 0.001606489, 0.0657686168, -0.01418446588, -0.020819385439999998, 0.028170218000000004, 0.01886695, 0.01504449513, -0.0070878272, -0.0322172944, -0.0206016932, 0.00234304962, 0.030583046200000002, 0.0743000296, -0.0002355583999999987, 0.020238522, 0.023775815, 0.0091016066, 0.007590398199999998, -0.00016460780000000178, 0.0022811768, 0.031100599600000004, -0.004822554799999998, -0.0027298507400000003, 0.0666423716, 0.0185833074, 0.0006633795999999997, 0.0174695822, 0.012306355599999999, 0.0025765082999999987,

In [ ]:
# 基于文本块的检索
persist_dir = '/home/jovyan/my_code/KG2RAG/output/tets_index'
if persist_dir and os.path.exists(persist_dir):
    print('Loading existing index...')
    sc = StorageContext.from_defaults(persist_dir=persist_dir)
    index = load_index_from_storage(sc)
else:
    print('Creating new index...')
    index = VectorStoreIndex(doc_chunks, show_progress=True)
    if persist_dir:
        os.makedirs(persist_dir, exist_ok=True)
        index.storage_context.persist(persist_dir=persist_dir)

Loading existing index...


In [ ]:
retriever = VectorIndexRetriever(index=index, similarity_top_k=top_k)

In [40]:
# 定义 RAG 问答模板
qa_rag_template_str = 'Context information is below.\n{context_str}\nQ: {query_str}\nA: '
qa_rag_prompt_template = PromptTemplate(qa_rag_template_str)

response_synthesizer = get_response_synthesizer(response_mode=ResponseMode.COMPACT, text_qa_template=qa_rag_prompt_template)

In [42]:
chunks_index

{'pmid32760437': 'Title: The accuracy of delirium assessment by cardiologists treating heart failure inpatients: a single center retrospective survey. Abstract: BACKGROUND: Patients with heart failure (HF) accompanied by delirium are at risk of rehospitalization and death, thus early detection and appropriate treatment is imperative. Palliative care for patients with HF is an important issue, particularly for patients who also have delirium. This retrospective study examined the accuracy of delirium assessment by cardiologists treating patients with HF, identified factors related to the detection of delirium, and recorded the initial treatment. METHODS: This was a retrospective chart survey of 165 patients with HF referred to a consultation liaison (C-L) service during treatment in the cardiology wards of a general hospital over a 6-year period. Diagnosis of delirium by the C-L psychiatrists was based on DSM-IV-TR.Cases in which cardiologists had stated "delirium" in the medical record

In [43]:
# 初始化知识图谱增强的后处理器
kg_post_processor1 = KGRetrievePostProcessor(ents=ents, doc2kg=doc2kg, chunks_index=chunks_index)

In [44]:
bge_reranker = FlagReranker(model_name_or_path=reranker)

In [45]:
kg_post_processor2 = GraphFilterPostProcessor(topk=top_k, ents=ents, doc2kg=doc2kg, chunks_index=chunks_index, reranker=bge_reranker)

In [47]:
engine = RetrieverQueryEngine(
    retriever=retriever,  #向量检索器
    response_synthesizer=response_synthesizer,   # 生成答案的组件, 定义如何组合检索到的信息
    node_postprocessors=[
        kg_post_processor1, # 利用知识图谱增强答案
        kg_post_processor2, # 使用 bge-reranker 重新排序答案。
        NaivePostprocessor()  # 个简单的文本清理后处理器。
        ]
    )

In [48]:
# question = input("请输入你的问题: ")

In [49]:
# 运行查询
question ="what chemical related to delirium?"
response = engine.query(question)
answer = response.response
sps = [source_node.node.id_ for source_node in response.source_nodes]

print("\n🔍 **Question:**", question)
print("📌 **Answer:**", answer)
print("📚 **Supporting Passages:**", sps)

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.



🔍 **Question:** what chemical related to delirium?
📌 **Answer:** <think>
Okay, I need to figure out which chemicals are related to delirium. Let me start by looking at the provided context information from the given PMIDs. 

First, I'll go through each abstract one by one.

1. **pmid36607634**: This is a systematic review about predisposing and precipitating factors associated with delirium. It mentions that there's no accepted method to group delirium into distinct subgroups but lists 33 predisposing and 112 precipitating factors across all clinical settings. However, it doesn't specify any particular chemicals.

2. **pmid32507939**: This study looks at the association between emergence delirium in elderly patients after general anesthesia. It uses specific assessment methods but doesn't mention any specific chemicals causing delirium.

3. **pmid36597428**: This is a quality improvement project about health literacy regarding delirium in older adults and their caregivers. The focus i

In [ ]:
# 运行查询
question ="what gene related to delirium?"
response = engine.query(question)
answer = response.response
sps = [source_node.node.id_ for source_node in response.source_nodes]

print("\n🔍 **Question:**", question)
print("📌 **Answer:**", answer)
print("📚 **Supporting evidence:**", sps)


🔍 **Question:** what gene related to delirium?
📌 **Answer:** <think>
Alright, so I need to figure out what genes are related to delirium based on the provided context information. Let me start by going through each of the abstracts one by one.

The first abstract (pmid31997329) talks about early recognition and biomarkers of delirium after cardiac surgery. It mentions that they looked for biomarkers, which might include genetic factors, but the abstract doesn't specify any particular genes. So this source isn't helpful in identifying specific genes related to delirium.

The second study (pmid36597428) is about health literacy regarding delirium in older adults and their caregivers. It discusses an educational intervention improving knowledge about delirium, but there's no mention of genetics or biomarkers here either. So this one doesn't help with the gene question.

The third abstract (pmid32507939) is a prospective study looking at emergence delirium and its association with postope